In [ ]:
# Upload dataset.csv from your computer
from google.colab import files
uploaded = files.upload()  # Click "Choose Files" and select dataset.csv

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib
import json

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
# Load the dataset
df = pd.read_csv('dataset.csv')

# Clean whitespace from all cells
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Get symptom columns (Symptom_1 to Symptom_17)
symptom_cols = [col for col in df.columns if col.startswith('Symptom')]

# Extract all unique symptoms
all_symptoms = set()
for col in symptom_cols:
    all_symptoms.update(df[col].dropna().unique())
all_symptoms.discard('')
all_symptoms = sorted(list(all_symptoms))

print(f"📋 Loaded {len(df)} rows")
print(f"🦠 Diseases: {df['Disease'].nunique()}")
print(f"🩺 Unique Symptoms: {len(all_symptoms)}")
print(f"📌 First 10 symptoms: {all_symptoms[:10]}")

📋 Loaded 4920 rows
🦠 Diseases: 41
🩺 Unique Symptoms: 131
📌 First 10 symptoms: ['abdominal_pain', 'abnormal_menstruation', 'acidity', 'acute_liver_failure', 'altered_sensorium', 'anxiety', 'back_pain', 'belly_pain', 'blackheads', 'bladder_discomfort']


In [ ]:
# Build binary feature vector for each row
def build_feature_vector(row):
    vector = np.zeros(len(all_symptoms), dtype=int)
    for col in symptom_cols:
        symptom = row[col]
        if pd.notna(symptom) and symptom in all_symptoms:
            vector[all_symptoms.index(symptom)] = 1
    return vector

# Create feature matrix X and labels y
X = np.array([build_feature_vector(row) for _, row in df.iterrows()])
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Disease'])

print(f"Feature matrix shape: {X.shape}")

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with 100 trees
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Test Accuracy: {accuracy * 100:.2f}%")

cv_scores = cross_val_score(model, X, y, cv=5)
print(f"📊 Cross-Validation: {cv_scores.mean() * 100:.2f}% (±{cv_scores.std() * 100:.2f}%)")

Feature matrix shape: (4920, 131)

🎯 Test Accuracy: 100.00%
📊 Cross-Validation: 100.00% (±0.00%)


In [ ]:
# Detailed per-disease performance
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        24
                                   AIDS       1.00      1.00      1.00        24
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        24
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        24
                       Bronchial Asthma       1.00      1.00      1.00        24
                   Cervical spondylosis       1.00      1.00      1.00        24
                            Chicken pox       1.00      1.00      1.00        24
                    Chronic cholestasis       1.00      1.00      1.00        24
                            Common Cold       1.00      1.00      1.00        24
                           

In [ ]:
# Save model artifacts
joblib.dump(model, 'model.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')

with open('symptom_columns.json', 'w') as f:
    json.dump(all_symptoms, f, indent=2)

print("✅ model.joblib saved")
print("✅ label_encoder.joblib saved")
print("✅ symptom_columns.json saved")

# Download all files to your PC
from google.colab import files
files.download('model.joblib')
files.download('label_encoder.joblib')
files.download('symptom_columns.json')

✅ model.joblib saved
✅ label_encoder.joblib saved
✅ symptom_columns.json saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>